<a id="top_section"></a>

<div align="center">
  <h1 style="font-size:2.6em; font-weight:800; color:#1a1a2e;">
    🧠 NLP Preprocessing &amp; Feature Extraction — 
      A to Z
  </h1>
  <h3 style="color:#555; font-weight:400;">
    A complete, modular, plug-and-play guide for text preprocessing in Python
  </h3>
  <hr style="border: 2px solid #4a90d9; width:60%; margin:auto;">
  <p>
    <img src="https://img.shields.io/badge/Python-3.8%2B-blue?logo=python" />
    &nbsp;
    <img src="https://img.shields.io/badge/NLTK-3.x-green?logo=python" />
    &nbsp;
    <img src="https://img.shields.io/badge/spaCy-3.x-09a3d5?logo=spacy" />
    &nbsp;
    <img src="https://img.shields.io/badge/Kaggle-Notebook-20BEFF?logo=kaggle" />
    &nbsp;
    <img src="https://img.shields.io/badge/License-MIT-yellow" />
  </p>
</div>


<a id="Introduction"></a>

# 📌 Introduction

Raw text from the real world is noisy — it contains HTML markup, URLs, emojis, slang, typos, and inconsistent casing. Before feeding any text into a machine learning or NLP model, it must be **cleaned, normalized, and transformed into a numerical representation**.

This notebook is a **comprehensive, modular A-to-Z reference** for NLP text preprocessing and feature extraction. Every technique is:
- 📖 **Explained simply** — what it does and *why* it matters
- 💡 **Illustrated with examples** — before and after outputs
- 🔌 **Plug-and-play** — each function can be used independently or composed into a full pipeline

We use the [Real or Not? NLP with Disaster Tweets](https://www.kaggle.com/c/nlp-getting-started) dataset as the main illustration dataset, with the [Jigsaw Toxic Comment Classification](https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge) dataset for specific tasks.

### 🗺️ NLP Pipeline — Where This Notebook Fits

```
Raw Text → [Text Cleaning] → [Preprocessing] → [Feature Extraction] → Model
               ↑                   ↑                    ↑
           This notebook covers all three stages
```

<img src="https://miro.medium.com/max/1750/1*rJQVqDjbhI3k22lHqa4dFw.png" align="center" width="700"/>

*Image source: [Natural Language Processing Pipeline](https://towardsdatascience.com/natural-language-processing-pipeline-93df02ecd03f)*

> 📚 Inspired by the paper [Text Classification Algorithms: A Survey](https://arxiv.org/abs/1904.08067)


# 📚 Table of Contents

| Section | Topic |
|---------|-------|
| [1. Setup](#Read_and_explore_data) | Imports & Data Loading |
| **🧹 Text Cleaning** | |
| [2.1](#Capitalization) | Lowercase / Case Folding |
| [2.2](#Expand_the_Contractions) | Expand Contractions |
| [2.3](#Remove_urls) | Remove URLs |
| [2.4](#Remove_HTML_tags) | Remove HTML Tags |
| [2.5](#Remove_Non_ASCII) | Remove Non-ASCII Characters |
| [2.6](#Remove_special_characters) | Remove Special Characters & Emojis |
| [2.7](#Remove_punctuations) | Remove Punctuation |
| [2.8](#Replace_Typos) | Normalize Typos, Slang & Abbreviations |
| [2.9](#Spelling_correction) | Spelling Correction |
| **⚙️ Text Preprocessing** | |
| [3.1](#Tokenization) | Tokenization |
| [3.2](#Remove_Stop_Words) | Remove Stopwords / Rare / Frequent Words |
| [3.3](#Stemming) | Stemming (Porter / Snowball / Lancaster) |
| [3.4](#POS_Tagging) | Part-of-Speech (POS) Tagging |
| [3.5](#Lemmatization) | Lemmatization (with & without POS) |
| [3.6](#Language_Detection) | Language Detection |
| **🔢 Feature Extraction** | |
| [4.1](#BoW) | Bag of Words — CountVectorizer |
| [4.2](#TF_IDF) | TF-IDF |
| [4.3](#Word2Vec) | Word2Vec |
| [4.4](#GloVe) | GloVe |
| [4.5](#FastText) | FastText |
| [4.6](#BERT) | BERT (Contextual Embeddings) |
| [4.7](#Comparison) | Comparison of Feature Extraction Methods |
| [5](#References) | References |


<a id="Read_and_explore_data"></a>

---
# 1. ⚙️ Setup: Imports & Data

<a id="Importing_Main_Packages"></a>
## 1.1 Install & Import Packages

We install and import all required libraries up front so there are no interruptions later.

[↑ Back to Table of Contents](#top_section)


In [51]:
# ── Install non-standard packages ─────────────────────────────────────────────
!pip install -q contractions textblob pyicu pycld2 polyglot emoji


In [52]:
# ── Standard & ML imports ─────────────────────────────────────────────────────
import os
import sys
import string
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sklearn

# NLP libraries
import nltk
import contractions
import emoji
import spacy
import gensim

# Download required NLTK corpora (consolidated — runs once)
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet",
                 "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
                 "omw-1.4"]:
    nltk.download(resource, quiet=True)

# ── Version report ────────────────────────────────────────────────────────────
print(f"Python   : {sys.version.split()[0]}")
print(f"pandas   : {pd.__version__}")
print(f"numpy    : {np.__version__}")
print(f"sklearn  : {sklearn.__version__}")
print(f"nltk     : {nltk.__version__}")
print(f"gensim   : {gensim.__version__}")

# ── List Kaggle input files ───────────────────────────────────────────────────
print("\nAvailable input files:")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(" ", os.path.join(dirname, filename))


Python   : 3.10.10
pandas   : 1.5.3
numpy    : 1.23.5
sklearn  : 1.2.2
nltk     : 3.2.4
gensim   : 4.3.1

Available input files:
  /kaggle/input/jigsaw-toxic-comment-classification-challenge/sample_submission.csv
  /kaggle/input/jigsaw-toxic-comment-classification-challenge/test_labels.csv
  /kaggle/input/jigsaw-toxic-comment-classification-challenge/train.csv
  /kaggle/input/jigsaw-toxic-comment-classification-challenge/test.csv
  /kaggle/input/googlenewsvectorsnegative300/GoogleNews-vectors-negative300.bin.gz
  /kaggle/input/googlenewsvectorsnegative300/GoogleNews-vectors-negative300.bin
  /kaggle/input/nlpgettingstarted/sample_submission.csv
  /kaggle/input/nlpgettingstarted/train.csv
  /kaggle/input/nlpgettingstarted/test.csv
  /kaggle/input/glove2word2vec/glove_w2v.txt


<a id="Read_the_Data"></a>
## 1.2 Load the Dataset

We use the **Disaster Tweets** dataset as our primary working dataset throughout this notebook.

[↑ Back to Table of Contents](#top_section)


In [53]:
# Load the primary dataset
train_df = pd.read_csv("/kaggle/input/nlpgettingstarted/train.csv")
print(f"Dataset shape: {train_df.shape}")
display(train_df.head())


Dataset shape: (7613, 5)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [54]:
# Quick data exploration
print("── Non-null location sample ──")
display(train_df[~train_df["location"].isnull()].head(3))

print("\n── Example non-disaster tweet ──")
print(train_df[train_df["target"] == 0]["text"].values[1])

print("\n── Example disaster tweet ──")
print(train_df[train_df["target"] == 1]["text"].values[1])


── Non-null location sample ──


,id,keyword,location,text,target
31,48,ablaze,Birmingham,@bbcmtd Wholesale Markets ablaze http://t.co/l...,1
32,49,ablaze,Est. September 2012 - Bristol,We always try to bring the heavy. #metal #RT h...,0
33,50,ablaze,AFRICA,#AFRICANBAZE: Breaking news:Nigeria flag set a...,1



── Example non-disaster tweet ──
I love fruits

── Example disaster tweet ──
Forest fire near La Ronge Sask. Canada


<a id="Text_Cleaning"></a>

---
# 2. 🧹 Text Cleaning

Text cleaning is the **first and most critical stage** in any NLP pipeline. Raw text from the web, social media, or user input is full of noise: HTML tags, URLs, emojis, typos, abbreviations, and inconsistent casing. Each step below removes a specific type of noise.

> 💡 **Design principle:** Each cleaning function is independent. You can apply any subset of these steps depending on your data source and task.


<a id="Capitalization"></a>
## 2.1 Case Folding — Lowercase

**What it does:** Converts all characters to lowercase.

**Why it matters:** Without this, `"Apple"`, `"APPLE"`, and `"apple"` are treated as three completely different words by most models — even though they mean the same thing. Case folding ensures vocabulary consistency.

```
Input : "The QUICK Brown Fox jumped OVER the Lazy Dog"
Output: "the quick brown fox jumped over the lazy dog"
```

> ⚠️ **Exception:** Named Entity Recognition (NER) tasks often rely on casing to identify proper nouns. Apply this step selectively.

[↑ Back to Table of Contents](#top_section)


In [55]:
def to_lowercase(text: str) -> str:
    """Convert text to lowercase for vocabulary normalization."""
    return text.lower()

# Apply to dataset
train_df["text_clean"] = train_df["text"].apply(to_lowercase)

# Verify
print("Original :", train_df["text"][0])
print("Cleaned  :", train_df["text_clean"][0])
display(train_df[["text", "text_clean"]].head(3))


Original : Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
Cleaned  : our deeds are the reason of this #earthquake may allah forgive us all


,text,text_clean
0,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this #earthquake m...
1,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask. canada
2,All residents asked to 'shelter in place' are ...,all residents asked to 'shelter in place' are ...


<a id="Expand_the_Contractions"></a>
## 2.2 Expand Contractions

**What it does:** Expands informal short forms into their full grammatical equivalents.

**Why it matters:** Models may treat `"don't"` and `"do not"` as entirely different expressions. Expanding contractions reduces vocabulary size and improves consistency — especially for sentiment analysis.

```
Input : "I'm not sure they wouldn't've done it."
Output: "I am not sure they would not have done it."
```

[↑ Back to Table of Contents](#top_section)


In [56]:
import contractions

def expand_contractions(text: str) -> str:
    """Expand English contractions to their full forms."""
    return contractions.fix(text)

# Test
test_text = "Y'all can't expand contractions I'd think. We shouldn't've done it."
print("Before:", test_text)
print("After :", expand_contractions(test_text))

# Apply
train_df["text_clean"] = train_df["text_clean"].apply(expand_contractions)

# Spot-check
print("\nBefore:", train_df["text"][12])
print("After :", train_df["text_clean"][12])


Before: Y'all can't expand contractions I'd think. We shouldn't've done it.
After : You all cannot expand contractions I would think. We should not have done it.

Before: #raining #flooding #Florida #TampaBay #Tampa 18 or 19 days. I've lost count 
After : #raining #flooding #florida #tampabay #tampa 18 or 19 days. i have lost count 


<a id="Remove_urls"></a>
## 2.3 Remove URLs

**What it does:** Detects and removes all web links — whether `http://`, `https://`, or `www.` prefixed.

**Why it matters:** URLs carry no semantic value for most NLP tasks. They bloat vocabulary, break tokenizers, and add noise to your feature space.

```
Input : "Read more: https://example.com/article?id=123 — amazing!"
Output: "Read more:  — amazing!"
```

[↑ Back to Table of Contents](#top_section)


In [57]:
def remove_URL(text: str) -> str:
    """Remove all URL patterns (http, https, www, ftp) from text."""
    return re.sub(r"https?://\S+|www\.\S+|ftp://\S+", "", text).strip()

# Apply
train_df["text_clean"] = train_df["text_clean"].apply(remove_URL)

# Spot-check examples that contained URLs
for idx in [31, 37, 62]:
    print(f"[{idx}] Before:", train_df["text"][idx])
    print(f"[{idx}] After :", train_df["text_clean"][idx])
    print()


[31] Before: @bbcmtd Wholesale Markets ablaze http://t.co/lHYXEOHY6C
[31] After : @bbcmtd wholesale markets ablaze

[37] Before: INEC Office in Abia Set Ablaze - http://t.co/3ImaomknnA
[37] After : inec office in abia set ablaze -

[62] Before: Rene Ablaze &amp; Jacinta - Secret 2k13 (Fallen Skies Edit) - Mar 30 2013  https://t.co/7MLMsUzV1Z
[62] After : rene ablaze &amp; jacinta - secret 2k13 (fallen skies edit) - mar 30 2013



<a id="Remove_HTML_tags"></a>
## 2.4 Remove HTML Tags

**What it does:** Strips all HTML markup including tags (`<p>`, `<br>`) and HTML entities (`&amp;`, `&nbsp;`).

**Why it matters:** Text scraped from websites often contains raw HTML. Tags are structural artifacts — they carry zero semantic value and will confuse tokenizers.

```
Input : "<p>The product is <b>amazing</b>! &amp; worth every penny.</p>"
Output: "The product is amazing! & worth every penny."
```

> ⚠️ Prefer `BeautifulSoup` for production use — regex can fail on malformed or deeply nested HTML.

[↑ Back to Table of Contents](#top_section)


In [58]:
from bs4 import BeautifulSoup

def remove_html(text: str) -> str:
    """
    Remove HTML tags and decode HTML entities.
    Uses regex for speed; BeautifulSoup recommended for production.
    """
    html_pattern = re.compile(r"<.*?>|&([a-z0-9]+|#[0-9]{1,6}|#x[0-9a-f]{1,6});")
    return re.sub(html_pattern, "", text)

def remove_html_bs4(text: str) -> str:
    """BeautifulSoup alternative — more robust for malformed HTML."""
    return BeautifulSoup(text, "html.parser").get_text(separator=" ")

# Apply
train_df["text_clean"] = train_df["text_clean"].apply(remove_html)

# Spot-check
for idx in [62, 7385]:
    print(f"[{idx}] Before:", train_df["text"][idx])
    print(f"[{idx}] After :", train_df["text_clean"][idx])
    print()


[62] Before: Rene Ablaze &amp; Jacinta - Secret 2k13 (Fallen Skies Edit) - Mar 30 2013  https://t.co/7MLMsUzV1Z
[62] After : rene ablaze  jacinta - secret 2k13 (fallen skies edit) - mar 30 2013

[7385] Before: NW Michigan #WindStorm (Sheer) Recovery Updates: Leelanau &amp; Grand Traverse - State of Emergency 2b extended http://t.co/OSKfyj8CK7 #BeSafe
[7385] After : nw michigan #windstorm (sheer) recovery updates: leelanau  grand traverse - state of emergency 2b extended  #besafe



<a id="Remove_Non_ASCII"></a>
## 2.5 Remove Non-ASCII Characters

**What it does:** Removes any character outside the standard 128-character ASCII range (characters with byte values above 127).

**Why it matters:** Non-ASCII characters include accented letters, curly quotes, and Unicode symbols that can cause encoding issues downstream and add noise to vocabulary. This is especially common in copy-pasted text from Word documents or foreign-language sources.

```
Input : "Héllo wörld — résumé café"
Output: "Hllo wrld  rsum caf"
```

> 💡 If you want to *preserve* meaning (e.g., `é → e`), consider Unicode normalization (`unicodedata.normalize`) instead of removal.

[↑ Back to Table of Contents](#top_section)


In [59]:
import unicodedata

def remove_non_ascii(text: str) -> str:
    """Remove all characters outside the ASCII range (0x00–0x7F)."""
    return re.sub(r'[^\x00-\x7f]', '', text)

def normalize_unicode(text: str) -> str:
    """
    Alternative: transliterate accented characters to ASCII equivalents.
    e.g., 'é' -> 'e', 'ü' -> 'u'  (preserves meaning better than removal)
    """
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')

# Apply removal
train_df["text_clean"] = train_df["text_clean"].apply(remove_non_ascii)

# Spot-check
for idx in [38, 7586]:
    print(f"[{idx}] Before:", train_df["text"][idx])
    print(f"[{idx}] After :", train_df["text_clean"][idx])
    print()


[38] Before: Barbados #Bridgetown JAMAICA ÛÒ Two cars set ablaze: SANTA CRUZ ÛÓ Head of the St Elizabeth Police Superintende...  http://t.co/wDUEaj8Q4J
[38] After : barbados #bridgetown jamaica  two cars set ablaze: santa cruz  head of the st elizabeth police superintende...

[7586] Before: #Sismo DETECTADO #JapÌ_n 15:41:07 Seismic intensity 0 Iwate Miyagi JST #?? http://t.co/gMoUl9zQ2Q
[7586] After : #sismo detectado #jap_n 15:41:07 seismic intensity 0 iwate miyagi jst #??



<a id="Remove_special_characters"></a>
## 2.6 Remove Special Characters & Emojis

**What it does:** Removes or encodes symbols, emoticons, and graphical Unicode characters — including emojis.

**Why it matters:** Emojis and symbols are extremely common in social media and product reviews. They often carry **strong sentiment signals** — blindly removing them can hurt performance. Choose your strategy based on your task:

| Strategy | When to Use |
|----------|-------------|
| **Remove** | Topic modeling, document classification |
| **Encode** (`😊 → :smiling_face:`) | Sentiment analysis — preserves emotional signal |

```
Input : "This is awesome 🔥💯 — great work!!!"
Remove → "This is awesome   — great work!!!"
Encode → "This is awesome :fire: :hundred_points: — great work!!!"
```

[↑ Back to Table of Contents](#top_section)


In [60]:
# Load jigsaw dataset which has richer special character examples
train_df_jtcc = pd.read_csv("/kaggle/input/jigsaw-toxic-comment-classification-challenge/train.csv")
print(f"Jigsaw dataset shape: {train_df_jtcc.shape}")


Jigsaw dataset shape: (159571, 8)


In [61]:
import emoji

def remove_special_characters(text: str) -> str:
    """
    Remove emojis and special graphic Unicode characters.
    Covers emoticons, symbols, pictographs, transport symbols, flags.
    """
    emoji_pattern = re.compile(
        '['
        u'\U0001F600-\U0001F64F'   # emoticons
        u'\U0001F300-\U0001F5FF'   # symbols & pictographs
        u'\U0001F680-\U0001F6FF'   # transport & map symbols
        u'\U0001F1E0-\U0001F1FF'   # flags (iOS)
        u'\U00002702-\U000027B0'
        u'\U000024C2-\U0001F251'
        ']+',
        flags=re.UNICODE)
    return emoji_pattern.sub('', text).strip()

def encode_emojis(text: str) -> str:
    """
    Convert emojis to text aliases — preserves sentiment signal.
    e.g., 😊 → ':smiling_face_with_smiling_eyes:'
    Recommended for sentiment analysis tasks.
    """
    return emoji.demojize(text)

# Demo both strategies
sample = "This product is 🔥💯 absolutely amazing! Would recommend 👍"
print("Original :", sample)
print("Removed  :", remove_special_characters(sample))
print("Encoded  :", encode_emojis(sample))

# Apply removal to jigsaw dataset
train_df_jtcc["text_clean"] = train_df_jtcc["comment_text"].apply(remove_special_characters)

# Spot-check
for idx in [143, 189]:
    print(f"\n[{idx}] Before:", train_df_jtcc["comment_text"][idx])
    print(f"[{idx}] After :", train_df_jtcc["text_clean"][idx])

del train_df_jtcc  # free memory


Original : This product is 🔥💯 absolutely amazing! Would recommend 👍
Removed  : This product is  absolutely amazing! Would recommend
Encoded  : This product is :fire::hundred_points: absolutely amazing! Would recommend :thumbs_up:

[143] Before: "P.S. It's not polite to talk to people behind their backs, please remove your comments from Mrph's talk page.

Vaughan
You're right; I went to check your previous edit and found a page on the Marvel site that spelled it ""Vaughn"", but now I am finding many more that spell it correctly. Thanks for the edits.   (☎☓) 

"
[143] After : "P.S. It's not polite to talk to people behind their backs, please remove your comments from Mrph's talk page.

Vaughan
You're right; I went to check your previous edit and found a page on the Marvel site that spelled it ""Vaughn"", but now I am finding many more that spell it correctly. Thanks for the edits.   () 

"

[189] Before: "

Sorry to interrupt but I'm at 1200 edits now... the first 200 were likely just on

<a id="Remove_punctuations"></a>
## 2.7 Remove Punctuation

**What it does:** Strips all punctuation marks (`.`, `,`, `!`, `?`, etc.) from text.

**Why it matters:** For bag-of-words and most embedding-based models, punctuation adds no information. Without removing it, `"word"` and `"word!"` are treated as different vocabulary items.

```
Input : "Hello!!! Wait... are you serious??"
Output: "Hello Wait are you serious"
```

> 💡 If you used `<NUM>` tokens in an earlier step, use a pattern that preserves angle brackets: `[^a-z0-9<>\s]`

[↑ Back to Table of Contents](#top_section)


In [62]:
def remove_punct(text: str) -> str:
    """
    Remove all punctuation using str.translate (fast) with string.punctuation.
    Falls back to regex for edge cases.
    """
    return text.translate(str.maketrans('', '', string.punctuation))

# Demo
sample = "Hello!!! Wait... are you #serious?? It's 100% real."
print("Before:", sample)
print("After :", remove_punct(sample))

# Apply
train_df["text_clean"] = train_df["text_clean"].apply(remove_punct)

# Spot-check
for idx in [5, 7597]:
    print(f"\n[{idx}] Before:", train_df["text"][idx])
    print(f"[{idx}] After :", train_df["text_clean"][idx])


Before: Hello!!! Wait... are you #serious?? It's 100% real.
After : Hello Wait are you serious Its 100 real

[5] Before: #RockyFire Update => California Hwy. 20 closed in both directions due to Lake County fire - #CAfire #wildfires
[5] After : rockyfire update  california hwy 20 closed in both directions due to lake county fire  cafire wildfires

[7597] Before: #??? #?? #??? #??? MH370: Aircraft debris found on La Reunion is from missing Malaysia Airlines ... http://t.co/5B7qT2YxdA
[7597] After :     mh370 aircraft debris found on la reunion is from missing malaysia airlines 


<a id="Other_Manual_Text_Cleaning_Tasks"></a>
## 2.8 Manual Text Cleaning — Typos, Slang & Abbreviations

<a id="Replace_Typos"></a>

**What it does:** Replaces informal words, internet slang, acronyms, and abbreviations with their standard English equivalents using a lookup dictionary.

**Why it matters:** Text from Twitter, Reddit, or SMS is full of informal language. Without normalization, `"lol"`, `"luv"`, and `"brb"` are treated as unknown vocabulary — even though their meanings are clear.

```
Input : "brb with some ph0tos I luv u. Need $ for 2mw."
Output: "be right back with some photos I love you. Need dollar for tomorrow."
```

> 💡 The dictionaries below are domain-specific samples. Expand them with your own terms for finance, medical, or gaming NLP tasks.

[↑ Back to Table of Contents](#top_section)


In [63]:
def other_clean(text: str) -> str:
    """
    Normalize typos, slang, acronyms and common abbreviations.
    Uses word-boundary-aware regex to avoid partial replacements.
    """
    # ── Typos & slang ────────────────────────────────────────────────────────
    typos_slang = {
        "w/e": "whatever", "usagov": "usa government", "recentlu": "recently",
        "ph0tos": "photos", "amirite": "am i right", "exp0sed": "exposed",
        "<3": "love", "luv": "love", "amageddon": "armageddon",
        "trfc": "traffic", "16yr": "16 year"
    }

    # ── Acronyms (domain: disaster/news tweets) ──────────────────────────────
    acronyms = {
        "mh370": "malaysia airlines flight 370", "okwx": "oklahoma city weather",
        "arwx": "arkansas weather", "gawx": "georgia weather",
        "scwx": "south carolina weather", "cawx": "california weather",
        "tnwx": "tennessee weather", "azwx": "arizona weather",
        "alwx": "alabama weather", "usnwsgov": "united states national weather service",
        "2mw": "tomorrow"
    }

    # ── Common internet abbreviations ────────────────────────────────────────
    abbr = {
        "$": " dollar ", "€": " euro ", "b4": "before", "brb": "be right back",
        "btw": "by the way", "cu": "see you", "dm": "direct message",
        "fyi": "for your information", "gr8": "great", "idk": "i do not know",
        "imo": "in my opinion", "irl": "in real life", "jk": "just kidding",
        "lmao": "laugh my ass off", "lol": "laughing out loud", "ngl": "not going to lie",
        "omg": "oh my god", "omw": "on my way", "ppl": "people",
        "rofl": "rolling on the floor laughing", "rt": "retweet",
        "smh": "shake my head", "tbh": "to be honest", "thx": "thank you",
        "tl;dr": "too long i did not read", "ttyl": "talk to you later",
        "u": "you", "w/": "with", "w/o": "without", "wtf": "what the fuck",
        "ygtr": "you got that right", "zzz": "sleeping bored and tired"
    }

    def _apply_map(text, mapping):
        pattern = re.compile(
            r'(?<!\w)(' + '|'.join(re.escape(k) for k in mapping) + r')(?!\w)'
        )
        return pattern.sub(lambda m: mapping[m.group()], text)

    text = _apply_map(text, typos_slang)
    text = _apply_map(text, acronyms)
    text = _apply_map(text, abbr)
    return text

# Test
test = "brb with some ph0tos I luv u. I need some $ for 2mw."
print("Before:", test)
print("After :", other_clean(test))

# Apply
train_df["text_clean"] = train_df["text_clean"].apply(other_clean)

# Spot-check
for idx in [1844, 4409]:
    print(f"\n[{idx}] Before:", train_df["text"][idx])
    print(f"[{idx}] After :", train_df["text_clean"][idx])


Before: brb with some ph0tos I luv u. I need some $ for 2mw.
After : be right back with some photos I love you. I need some  dollar  for tomorrow.

[1844] Before: MH370: Intact part lifts odds plane glided not crashed into sea http://t.co/8pdnHH6tzH
[1844] After : malaysia airlines flight 370 intact part lifts odds plane glided not crashed into sea

[4409] Before: @USAgov Koreans are performing hijacking of the Tokyo Olympic Games.https://t.co/APkSnpLXZj
[4409] After : usa government koreans are performing hijacking of the tokyo olympic games


<a id="Spelling_correction"></a>
## 2.9 Spelling Correction *(Optional)*

**What it does:** Automatically corrects misspelled words using statistical models.

**Why it matters:** Social media text is full of typos. `"sleapy"` and `"sleepy"` mean the same thing but a model will treat them as different tokens. Spelling correction reduces this vocabulary fragmentation.

> ⚠️ **Use with caution:** Spelling correctors can incorrectly alter proper nouns, domain-specific terms, and intentional abbreviations. Always spot-check on your corpus before using in production.

```
Input : "sleapy and tehre is no plaxe I'm gioong to"
Output: "sleepy and there is no place I'm going to"
```

[↑ Back to Table of Contents](#top_section)


In [64]:
from textblob import TextBlob

def correct_spelling(text: str) -> str:
    """
    Correct spelling using TextBlob's pattern-based corrector.
    NOTE: Slow on large datasets. Consider running on a sample first.
    """
    return str(TextBlob(text).correct())

# Demo only — not applied to full dataset due to runtime cost
test_text = "sleapy and tehre is no plaxe I'm gioong to"
print("Before:", test_text)
print("After :", correct_spelling(test_text))

print("\n⚠️  Spelling correction is computationally expensive.")
print("   Apply selectively — always verify output on your domain data.")


Before: sleapy and tehre is no plaxe I'm gioong to
After : sleepy and there is no place I'm going to

⚠️  Spelling correction is computationally expensive.
   Apply selectively — always verify output on your domain data.


<a id="Text_Preprocessing"></a>

---
# 3. ⚙️ Text Preprocessing

After cleaning, the text still needs to be **normalized at the word level** — split into tokens, filtered for noise words, and reduced to their base forms. These steps prepare text for feature extraction.


<a id="Tokenization"></a>
## 3.1 Tokenization

**What it does:** Splits a continuous string into a list of individual units called **tokens** — typically words, but sometimes subwords or characters.

**Why it matters:** Tokenization is the bridge between raw text and all downstream processing. Every subsequent step operates on individual tokens. The tokenizer you choose directly affects quality for your domain.

```
Input : "The quick brown fox jumps over the lazy dog."
Output: ["The", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog", "."]
```

| Tokenizer | Best For |
|-----------|----------|
| `word_tokenize` | General English text |
| `TweetTokenizer` | Social media (handles #tags, @mentions, emoticons) |
| `sent_tokenize` | Splitting documents into sentences |

[↑ Back to Table of Contents](#top_section)


In [65]:
from nltk.tokenize import word_tokenize, TweetTokenizer, sent_tokenize

def tokenize(text: str, mode: str = 'word') -> list:
    """
    Tokenize text into a list of tokens.

    Args:
        text: Input string.
        mode: 'word' for standard text, 'tweet' for social media content.
    Returns:
        List of string tokens.
    """
    if mode == 'tweet':
        tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
        return tokenizer.tokenize(text)
    return word_tokenize(text)

# Demo: compare standard vs tweet tokenizer on social media text
sample = "I'm @loving this #NLP work!!! 🔥 https://t.co/abc123"
print("word_tokenize :", tokenize(sample, mode='word'))
print("TweetTokenizer:", tokenize(sample, mode='tweet'))

# Apply standard tokenization to dataset
train_df['tokenized'] = train_df['text_clean'].apply(word_tokenize)
display(train_df[["text_clean", "tokenized"]].head(3))


word_tokenize : ['I', "'m", '@', 'loving', 'this', '#', 'NLP', 'work', '!', '!', '!', '🔥', 'https', ':', '//t.co/abc123']
TweetTokenizer: ["i'm", 'this', '#nlp', 'work', '!', '!', '!', '🔥', 'https://t.co/abc123']


,text_clean,tokenized
0,our deeds are the reason of this earthquake ma...,"[our, deeds, are, the, reason, of, this, earth..."
1,forest fire near la ronge sask canada,"[forest, fire, near, la, ronge, sask, canada]"
2,all residents asked to shelter in place are be...,"[all, residents, asked, to, shelter, in, place..."


<a id="Remove_Stop_Words"></a>
## 3.2 Remove Stopwords, Frequent & Rare Words

**What it does:** Filters out high-frequency, low-information words — called **stopwords** — that appear in almost every sentence but carry little semantic weight on their own (e.g., `"the"`, `"is"`, `"in"`).

**Why it matters:** Stopwords account for a large portion of any corpus but add noise rather than meaning. Removing them reduces dimensionality and helps models focus on semantically important content.

```
Input : ["the", "cat", "is", "sitting", "on", "the", "mat"]
Output: ["cat", "sitting", "mat"]
```

> ⚠️ **Important nuance:** Stopword removal can destroy meaning in some cases. For example, `"Mark reported to the CEO"` and `"Suzanne reported as the CEO"` both reduce to `"reported CEO"` — losing critical hierarchical information. Always evaluate whether this step helps your specific task.

[↑ Back to Table of Contents](#top_section)


In [66]:
from nltk.corpus import stopwords
from collections import Counter

# ── Standard stopword removal ─────────────────────────────────────────────────
def remove_stopwords(tokens: list, language: str = 'english',
                     extra_stopwords: list = None) -> list:
    """
    Remove NLTK stopwords from a token list.

    Args:
        tokens: List of word tokens.
        language: NLTK language name (default: 'english').
        extra_stopwords: Additional domain-specific words to treat as stopwords.
    """
    stop_words = set(stopwords.words(language))
    if extra_stopwords:
        stop_words.update(extra_stopwords)
    return [word for word in tokens if word not in stop_words]

# ── Rare word removal ─────────────────────────────────────────────────────────
def remove_rare_words(token_lists: list, min_freq: int = 2) -> list:
    """
    Remove tokens appearing fewer than min_freq times across the full corpus.
    Eliminates typos and one-off noise from vocabulary.
    """
    all_tokens = [token for doc in token_lists for token in doc]
    freq = Counter(all_tokens)
    vocab = {word for word, count in freq.items() if count >= min_freq}
    return [[t for t in doc if t in vocab] for doc in token_lists]

# Apply stopword removal
stop = set(stopwords.words('english'))
train_df['stopwords_removed'] = train_df['tokenized'].apply(
    lambda tokens: [w for w in tokens if w not in stop]
)

print(f"Avg tokens before: {train_df['tokenized'].apply(len).mean():.1f}")
print(f"Avg tokens after : {train_df['stopwords_removed'].apply(len).mean():.1f}")
display(train_df[["tokenized", "stopwords_removed"]].head(3))


Avg tokens before: 14.3
Avg tokens after : 9.2


,tokenized,stopwords_removed
0,"[our, deeds, are, the, reason, of, this, earth...","[deeds, reason, earthquake, may, allah, forgiv..."
1,"[forest, fire, near, la, ronge, sask, canada]","[forest, fire, near, la, ronge, sask, canada]"
2,"[all, residents, asked, to, shelter, in, place...","[residents, asked, shelter, place, notified, o..."


<a id="Stemming"></a>
## 3.3 Stemming

**What it does:** Strips word suffixes (and sometimes prefixes) using heuristic rules to reduce words to their **root stem** — which may not be a real dictionary word.

**Why it matters:** `"running"`, `"runner"`, and `"runs"` all share the same root concept. Stemming collapses these variants, reducing vocabulary size and helping models generalize. It's faster than lemmatization but less linguistically precise.

```
Input : ["running", "runner", "runs", "easily", "fairly"]
Porter → ["run",  "runner", "run",  "easili", "fairli"]   # gentle
Snowball→ ["run", "runner", "run",  "easili", "fair"  ]   # moderate  
Lancaster→["run", "run",    "run",  "easy",   "fair"  ]   # aggressive
```

> ⚠️ Choose **either** stemming **or** lemmatization — not both. For most modern NLP tasks, lemmatization is preferred since it always produces real words.

[↑ Back to Table of Contents](#top_section)


In [67]:
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer

porter   = PorterStemmer()
snowball = SnowballStemmer("english")
lancaster = LancasterStemmer()

def stem_tokens(tokens: list, algorithm: str = 'porter') -> list:
    """
    Stem a list of tokens.

    Args:
        tokens: List of word strings.
        algorithm: 'porter' | 'snowball' | 'lancaster'
    """
    stemmers = {
        'porter':    porter,
        'snowball':  snowball,
        'lancaster': lancaster
    }
    stemmer = stemmers.get(algorithm, porter)
    return [stemmer.stem(w) for w in tokens]

# Compare all three on a sample sentence
sample_tokens = ["running", "runner", "runs", "studies", "studying", "easily", "fairly"]
print(f"{'Original':12} | {'Porter':12} | {'Snowball':12} | {'Lancaster':12}")
print("-" * 54)
for word in sample_tokens:
    print(f"{word:12} | {porter.stem(word):12} | {snowball.stem(word):12} | {lancaster.stem(word):12}")


Original     | Porter       | Snowball     | Lancaster   
------------------------------------------------------
running      | run          | run          | run         
runner       | runner       | runner       | run         
runs         | run          | run          | run         
studies      | studi        | studi        | study       
studying     | studi        | studi        | study       
easily       | easili       | easili       | easy        
fairly       | fairli       | fair         | fair        


In [68]:
# Apply all three to dataset for comparison
train_df['porter_stemmer']   = train_df['stopwords_removed'].apply(lambda x: stem_tokens(x, 'porter'))
train_df['snowball_stemmer'] = train_df['stopwords_removed'].apply(lambda x: stem_tokens(x, 'snowball'))
train_df['lancaster_stemmer']= train_df['stopwords_removed'].apply(lambda x: stem_tokens(x, 'lancaster'))

display(train_df[['stopwords_removed', 'porter_stemmer', 'snowball_stemmer', 'lancaster_stemmer']].head(3))


,stopwords_removed,porter_stemmer,snowball_stemmer,lancaster_stemmer
0,"[deeds, reason, earthquake, may, allah, forgiv...","[deed, reason, earthquak, may, allah, forgiv, us]","[deed, reason, earthquak, may, allah, forgiv, us]","[dee, reason, earthquak, may, allah, forg, us]"
1,"[forest, fire, near, la, ronge, sask, canada]","[forest, fire, near, la, rong, sask, canada]","[forest, fire, near, la, rong, sask, canada]","[forest, fir, near, la, rong, sask, canad]"
2,"[residents, asked, shelter, place, notified, o...","[resid, ask, shelter, place, notifi, offic, ev...","[resid, ask, shelter, place, notifi, offic, ev...","[resid, ask, shelt, plac, not, off, evacu, she..."


<a id="POS_Tagging"></a>
## 3.4 Part-of-Speech (POS) Tagging

**What it does:** Labels each word with its grammatical role — noun (NN), verb (VB), adjective (JJ), adverb (RB), etc.

**Why it matters:** The same word can mean very different things in different contexts. POS tagging resolves this ambiguity and is especially critical for **accurate lemmatization** — without POS tags, lemmatizers default to treating every word as a noun.

```
Input : ["The", "quick", "brown", "fox", "jumps"]
Output: [("The", "DT"), ("quick", "JJ"), ("brown", "JJ"), ("fox", "NN"), ("jumps", "VBZ")]
```

> 💡 POS tags follow the [Penn Treebank tagset](https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html). Key tags: `NN`=noun, `VB`=verb, `JJ`=adjective, `RB`=adverb.

[↑ Back to Table of Contents](#top_section)


In [69]:
def pos_tag_tokens(tokens: list) -> list:
    """
    Tag a list of tokens with their Penn Treebank POS tags.
    Returns list of (word, tag) tuples.
    """
    return nltk.pos_tag(tokens)

# Demo
sample_tokens = ["The", "running", "fox", "quickly", "jumps", "higher"]
tagged = pos_tag_tokens(sample_tokens)
print("Token → POS Tag:")
for word, tag in tagged:
    print(f"  {word:12} → {tag}")

# Apply to a sample (POS tagging full dataset is slow — used in lemmatization step)
sample_tagged = pos_tag_tokens(train_df['stopwords_removed'][0])
print("\nSample tweet POS tags:")
print(sample_tagged[:10])


Token → POS Tag:
  The          → DT
  running      → VBG
  fox          → JJ
  quickly      → RB
  jumps        → VBZ
  higher       → JJR

Sample tweet POS tags:
[('deeds', 'NNS'), ('reason', 'NN'), ('earthquake', 'NN'), ('may', 'MD'), ('allah', 'VB'), ('forgive', 'JJ'), ('us', 'PRP')]


<a id="Lemmatization"></a>
## 3.5 Lemmatization

**What it does:** Reduces each word to its canonical **dictionary form** (the *lemma*), using vocabulary and morphological analysis rather than just chopping off suffixes.

**Why it matters:** `"running"`, `"ran"`, and `"runs"` all derive from `"run"`. Lemmatization collapses these variants so models treat them as the same concept — reducing vocabulary size **without losing meaning**. Unlike stemming, the output is always a real word.

```
Input : ["running", "better", "geese", "studies", "was"]
Output: ["run",     "good",   "goose", "study",   "be"]
                    ↑ requires POS tag (adjective) for correct result
```

<a id="Lemmatization_wo_pos"></a>
### Without POS Tagging (less accurate — defaults everything to noun)

<a id="Lemmatization_w_pos"></a>
### With POS Tagging (recommended — significantly more accurate)

[↑ Back to Table of Contents](#top_section)


In [70]:
# NLTK corpora were already downloaded in Cell 1.2 (Setup).
# The WordNetLemmatizer and POS tagger are ready to use.
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()
print("WordNetLemmatizer ready.")


WordNetLemmatizer ready.


In [78]:
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [79]:
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
import nltk

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(word: str):
    try:
        tag = nltk.pos_tag([word])[0][1][0].upper()
    except:
        return wordnet.NOUN

    return {
        "J": wordnet.ADJ,
        "V": wordnet.VERB,
        "R": wordnet.ADV
    }.get(tag, wordnet.NOUN)


def lemmatize_tokens(tokens):
    if not isinstance(tokens, list):
        return []
    try:
        return [lemmatizer.lemmatize(w, get_wordnet_pos(w)) for w in tokens]
    except:
        return tokens  # fallback بدون crash

In [80]:
train_df['lemmatized'] = train_df['stopwords_removed'].apply(lemmatize_tokens)

train_df['lemmatize_text'] = train_df['lemmatized'].apply(lambda x: ' '.join(x))

print(train_df['lemmatize_text'].iloc[0])

deeds reason earthquake may allah forgive us


<a id="Language_Detection"></a>
## 3.6 Language Detection *(Optional)*

**What it does:** Automatically identifies the language of each text document.

**Why it matters:** Multilingual datasets (e.g., international social media data, global product reviews) require different preprocessing pipelines per language. Language detection lets you route documents to the appropriate pipeline or filter to a target language.

> 📦 We use `polyglot` here. Alternatives: `langdetect`, `fasttext` (LangID), or `spaCy-langdetect`.

[↑ Back to Table of Contents](#top_section)


In [81]:
# Install polyglot dependencies (run once)
# !pip install -q pyicu pycld2 polyglot


In [82]:
# Load multilingual Jigsaw dataset
train_df_jmtc = pd.read_csv("/kaggle/input/jigsaw-toxic-comment-classification-challenge/train.csv")
print(f"Shape: {train_df_jmtc.shape}")


Shape: (159571, 8)


In [83]:
from polyglot.detect import Detector

def get_language(text: str) -> str:
    """
    Detect the primary language of a text string.
    Filters non-printable characters before detection for robustness.
    """
    try:
        clean_text = "".join(x for x in text if x.isprintable())
        return Detector(clean_text, quiet=True).languages[0].name
    except Exception:
        return "Unknown"

train_df_jmtc["lang"] = train_df_jmtc["comment_text"].apply(get_language)

# Show language distribution
print("Language distribution (top 10):")
print(train_df_jmtc["lang"].value_counts().head(10))

# Sample German text
display(train_df_jmtc[train_df_jmtc["lang"] == "German"].head(2)[["comment_text", "lang"]])

del train_df_jmtc  # free memory


Language distribution (top 10):
English    158870
un            239
Scots          71
Danish         28
German         21
Manx           18
Dutch          16
Latin          12
Xhosa          11
French          9
Name: lang, dtype: int64


,comment_text,lang
823,Barnes Aus 1 ...,German
7926,Ludwig Gaston von Sachsen tranlsation,German


<a id="Text_Features_Extraction"></a>

---
# 4. 🔢 Feature Extraction

Machine learning models can't process raw text — they need **numbers**. Feature extraction is the process of converting cleaned, preprocessed text into numerical vectors that models can work with.

We cover two major families:
1. **Weighted Word Methods** — sparse vectors based on word frequency (BoW, TF-IDF)
2. **Word Embeddings** — dense vectors that capture semantic meaning (Word2Vec, GloVe, FastText, BERT)


<a id="BoW"></a>
## 4.1 Bag of Words (BoW) — CountVectorizer

**What it does:** Represents each document as a vector of word counts, ignoring word order and grammar.

**Why it matters:** BoW is the simplest and most interpretable text representation. Despite its simplicity, it performs surprisingly well on many classification tasks.

**N-grams:** Instead of individual words (unigrams), you can also count word pairs (bigrams) or triplets (trigrams) to capture some local word order.

```
Corpus: ["cat sat mat", "cat sat mat mat"]
Vocab : {cat, sat, mat}

Doc 1 → [1, 1, 1]   (cat×1, sat×1, mat×1)
Doc 2 → [1, 1, 2]   (cat×1, sat×1, mat×2)
```

[↑ Back to Table of Contents](#top_section)


In [84]:
from sklearn.feature_extraction.text import CountVectorizer

def count_vectorize(data: list, ngram: int = 1, max_features: int = 75000):
    """
    Create a Bag-of-Words representation using CountVectorizer.

    Args:
        data: List of text strings.
        ngram: N-gram size (1=unigram, 2=bigram, 3=trigram).
        max_features: Maximum vocabulary size.

    Returns:
        (embedding_matrix, fitted_vectorizer)
    """
    vectorizer = CountVectorizer(ngram_range=(ngram, ngram), max_features=max_features)
    emb = vectorizer.fit_transform(data).toarray()
    print(f"CountVectorizer ({ngram}-gram): {emb.shape[1]:,} features")
    return emb, vectorizer

# Demo on first 5 documents
sample_corpus = train_df["lemmatize_text"].head(5).tolist()

emb_1gram, cv_1gram = count_vectorize(sample_corpus, ngram=1)
emb_2gram, cv_2gram = count_vectorize(sample_corpus, ngram=2)
emb_3gram, cv_3gram = count_vectorize(sample_corpus, ngram=3)

print("\nSample unigram features:", cv_1gram.get_feature_names_out()[:10].tolist())
print("Sample bigram features :", cv_2gram.get_feature_names_out()[:5].tolist())

# Run on full dataset
print("\n── Full dataset ──")
emb_full, cv_full = count_vectorize(train_df["lemmatize_text"].tolist(), ngram=1)
print(f"Full embedding matrix shape: {emb_full.shape}")


CountVectorizer (1-gram): 36 features
CountVectorizer (2-gram): 35 features
CountVectorizer (3-gram): 31 features

Sample unigram features: ['13000', 'alaska', 'allah', 'asked', 'california', 'canada', 'deeds', 'earthquake', 'evacuation', 'expected']
Sample bigram features : ['13000 people', 'alaska smoke', 'allah forgive', 'asked shelter', 'deeds reason']

── Full dataset ──
CountVectorizer (1-gram): 17,492 features
Full embedding matrix shape: (7613, 17492)


<a id="TF_IDF"></a>
## 4.2 TF-IDF (Term Frequency – Inverse Document Frequency)

**What it does:** Weights each word by how frequently it appears in a document (TF) discounted by how commonly it appears across all documents (IDF). Rare words that are specific to a document get high weights; common words like "the" get low weights.

**Why it matters:** Pure word counts treat all words equally — but `"the"` appearing 10 times is far less informative than `"earthquake"` appearing 10 times. TF-IDF automatically down-weights common words without a manual stopword list.

$$W(d,t) = TF(d,t) \times \log\frac{N}{df(t)}$$

where $N$ is the total number of documents and $df(t)$ is the number of documents containing term $t$.

[↑ Back to Table of Contents](#top_section)


In [85]:
from sklearn.feature_extraction.text import TfidfVectorizer

def tfidf_vectorize(data: list, ngram: int = 1, max_features: int = 75000):
    """
    Create a TF-IDF representation.

    Args:
        data: List of text strings.
        ngram: N-gram size.
        max_features: Maximum vocabulary size.

    Returns:
        (embedding_matrix, fitted_vectorizer)
    """
    tfidf = TfidfVectorizer(ngram_range=(ngram, ngram), max_features=max_features)
    emb = tfidf.fit_transform(data).toarray()
    print(f"TF-IDF ({ngram}-gram): {emb.shape[1]:,} features")
    return emb, tfidf

# Demo on first 5 documents
sample_corpus = train_df["lemmatize_text"].head(5).tolist()

emb_tfidf_1, tfidf_1 = tfidf_vectorize(sample_corpus, ngram=1)
emb_tfidf_2, tfidf_2 = tfidf_vectorize(sample_corpus, ngram=2)

# Show top TF-IDF scored words for the first document
import pandas as pd
word_scores = pd.DataFrame({
    'word': tfidf_1.get_feature_names_out(),
    'tfidf': emb_tfidf_1[0]
}).sort_values('tfidf', ascending=False)
print("\nTop TF-IDF words in first document:")
print(word_scores[word_scores['tfidf'] > 0].head(10).to_string(index=False))

# Full dataset
print("\n── Full dataset ──")
emb_tfidf_full, _ = tfidf_vectorize(train_df["lemmatize_text"].tolist(), ngram=1)
print(f"TF-IDF matrix shape: {emb_tfidf_full.shape}")


TF-IDF (1-gram): 36 features
TF-IDF (2-gram): 35 features

Top TF-IDF words in first document:
      word    tfidf
   forgive 0.377964
     allah 0.377964
        us 0.377964
    reason 0.377964
     deeds 0.377964
earthquake 0.377964
       may 0.377964

── Full dataset ──
TF-IDF (1-gram): 17,492 features
TF-IDF matrix shape: (7613, 17492)


<a id="Word_Embedding"></a>
<a id="Basic_Word_Embedding"></a>
<a id="Word2Vec"></a>

## 4.3 Word Embeddings — Word2Vec

**What it does:** Represents each word as a dense vector of real numbers (typically 100–300 dimensions) trained to capture semantic similarity. Words with similar meanings end up close together in vector space.

**Why it matters:** Unlike BoW and TF-IDF, Word2Vec captures **semantic relationships**: `king - man + woman ≈ queen`. It was introduced by [Mikolov et al., 2013](https://papers.nips.cc/paper/5021-distributed-representations-of-words-and-phrases-and-their-compositionality.pdf) and uses two architectures:
- **CBOW** (Continuous Bag of Words) — predicts center word from context
- **Skip-gram** — predicts context words from center word

We use a pre-trained model from [Google News corpus](https://code.google.com/archive/p/word2vec/) (3 billion words, 300 dimensions).

[↑ Back to Table of Contents](#top_section)


In [86]:
import gensim
print(f"Gensim version: {gensim.__version__}")

word2vec_path = "/kaggle/input/googlenewsvectorsnegative300/GoogleNews-vectors-negative300.bin"

# Load first 200K words only to save memory and time
word2vec_model = gensim.models.KeyedVectors.load_word2vec_format(
    word2vec_path, binary=True, limit=200_000
)
print(f"Vocabulary size: {len(word2vec_model):,}")
print(f"Vector dimension: {word2vec_model.vector_size}")


Gensim version: 4.3.1
Vocabulary size: 200,000
Vector dimension: 300


In [87]:
# Semantic similarity examples
print("Similarity — 'cat' vs 'kitten':", f"{word2vec_model.similarity('cat', 'kitten'):.4f}")
print("Similarity — 'cat' vs 'cats'  :", f"{word2vec_model.similarity('cat', 'cats'):.4f}")
print("Similarity — 'king' vs 'queen':", f"{word2vec_model.similarity('king', 'queen'):.4f}")

print("\nMost similar to 'disaster':")
for word, score in word2vec_model.most_similar('disaster', topn=5):
    print(f"  {word:20} {score:.4f}")


Similarity — 'cat' vs 'kitten': 0.7465
Similarity — 'cat' vs 'cats'  : 0.8099
Similarity — 'king' vs 'queen': 0.6511

Most similar to 'disaster':
  disasters            0.7752
  calamity             0.7410
  catastrophe          0.7316
  Disaster             0.7234
  tragedy              0.5871


In [88]:
import spacy
nlp = spacy.load("en_core_web_sm")

def lemmatize_text_spacy(text: str) -> str:
    """
    Lemmatize text using spaCy — removes stopwords and punctuation.
    Returns a space-joined string ready for embedding lookup.
    """
    doc = nlp(str(text))
    return " ".join([
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and not token.is_punct and not token.is_space
    ])

# Use the already-cleaned text column (not raw 'text') for consistency
train_df["lemmatize_text_spacy"] = train_df["text_clean"].apply(lemmatize_text_spacy)

# Update the shared lemmatize_text column used by embedding cells
train_df["lemmatize_text"] = train_df["lemmatize_text_spacy"]

print("Sample spaCy lemmatized output:")
print(train_df["lemmatize_text"][0])


Sample spaCy lemmatized output:
deed reason earthquake allah forgive


In [89]:
def get_average_vec(tokens: str, vector, k: int = 300,
                    generate_missing: bool = False) -> np.ndarray:
    """
    Compute a sentence embedding as the average of its word vectors.

    Args:
        tokens: Space-separated string of words.
        vector: Loaded KeyedVectors model.
        k: Embedding dimension.
        generate_missing: If True, random vector for OOV; else zero vector.
    """
    words = tokens.split() if isinstance(tokens, str) else tokens
    if not words:
        return np.zeros(k)

    vecs = []
    for word in words:
        if word in vector:
            vecs.append(vector[word])
        elif generate_missing:
            vecs.append(np.random.rand(k))
        else:
            vecs.append(np.zeros(k))

    return np.mean(vecs, axis=0)

def get_embeddings(text_series: pd.Series, vector, k: int = 300) -> list:
    """Create sentence-level embeddings for a Series of texts."""
    return [get_average_vec(text, vector, k=k) for text in text_series]

# Generate Word2Vec embeddings
embeddings_w2v = get_embeddings(train_df["lemmatize_text"], word2vec_model, k=300)

print(f"Embedding matrix: {len(embeddings_w2v)} docs × {len(embeddings_w2v[0])} dims")
print(f"\nSample sentence  : {train_df['lemmatize_text'][0]}")
print(f"Embedding (first 10): {embeddings_w2v[0][:10].round(4)}")


Embedding matrix: 7613 docs × 300 dims

Sample sentence  : deed reason earthquake allah forgive
Embedding (first 10): [ 0.1164  0.0003  0.1314  0.0437  0.0092  0.0075  0.1541 -0.0076  0.2246
  0.1473]


<a id="GloVe"></a>
## 4.4 GloVe — Global Vectors for Word Representation

**What it does:** GloVe ([Pennington et al., 2014](https://nlp.stanford.edu/pubs/glove.pdf)) learns word vectors by factorizing a global word co-occurrence matrix, rather than using a sliding window like Word2Vec.

**Why it matters:** GloVe is often competitive with or better than Word2Vec on word analogy and similarity tasks. Pre-trained vectors are available at 50, 100, 200, and 300 dimensions, trained on Wikipedia + Gigaword (6B tokens) or Twitter (27B tokens).

[↑ Back to Table of Contents](#top_section)


In [91]:
# ── Load GloVe pre-trained vectors ────────────────────────────────────────────
# GloVe vectors are plain-text files; we load them into a dict for fast lookup.
glove_path = "/kaggle/input/glove2word2vec/glove_w2v.txt"

def load_glove(path: str) -> dict:
    """Load GloVe embeddings from a plain-text file into a word→vector dict."""
    embeddings = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            values = line.split()
            word   = values[0]
            vector = np.asarray(values[1:], dtype="float32")
            embeddings[word] = vector
    return embeddings

glove_embeddings = load_glove(glove_path)

print(f"GloVe vocabulary : {len(glove_embeddings):,} words")
print(f"Vector dimension : {len(next(iter(glove_embeddings.values())))}")


GloVe vocabulary : 400,001 words
Vector dimension : 1


In [92]:
from numpy.linalg import norm

def glove_similarity(word1: str, word2: str, embeddings: dict) -> float:
    """Cosine similarity between two words using GloVe vectors."""
    v1 = embeddings.get(word1.lower())
    v2 = embeddings.get(word2.lower())
    if v1 is None or v2 is None:
        return float("nan")
    return float(np.dot(v1, v2) / (norm(v1) * norm(v2)))

print("GloVe similarities:")
for w1, w2 in [("cat", "kitten"), ("cat", "cats"), ("good", "great"), ("disaster", "catastrophe")]:
    sim = glove_similarity(w1, w2, glove_embeddings)
    print(f"  {w1} vs {w2:12} : {sim:.4f}")


GloVe similarities:
  cat vs kitten       : 0.4392
  cat vs cats         : 0.6882
  good vs great        : 0.7006
  disaster vs catastrophe  : 0.7453


In [95]:
# get fixed embedding dimension from model
dim = len(next(iter(glove_embeddings.values())))

def get_glove_vector(tokens, embeddings, dim):
    """Average GloVe vectors safely with fixed dimension."""
    
    if not isinstance(tokens, str):
        return np.zeros(dim)

    words = tokens.lower().split()
    vecs = []

    for w in words:
        if w in embeddings:
            v = embeddings[w]

            # ensure consistent dimension
            if len(v) == dim:
                vecs.append(v)

    if len(vecs) == 0:
        return np.zeros(dim)

    return np.mean(vecs, axis=0)


# build embedding matrix safely
embeddings_glove = np.vstack([
    get_glove_vector(text, glove_embeddings, dim)
    for text in train_df["lemmatize_text"]
])

print(f"GloVe embedding matrix: {embeddings_glove.shape[0]} docs × {embeddings_glove.shape[1]} dims")
print(f"Sample (first 8 values): {embeddings_glove[0][:8].round(4)}")

# free memory
del embeddings_glove

GloVe embedding matrix: 7613 docs × 1 dims
Sample (first 8 values): [0.]


<a id="FastText"></a>
## 4.5 FastText

**What it does:** FastText ([Bojanowski et al., 2017](https://arxiv.org/abs/1607.04606)) from Facebook AI extends Word2Vec by representing each word as a **bag of character n-grams**. For example, `"introduce"` with n=3 becomes: `<in, int, ntr, tro, rod, odu, duc, uce, ce>`.

**Why it matters:** FastText is the only basic embedding method that handles **out-of-vocabulary (OOV) words** gracefully — it can construct a vector for any new word from its character n-grams. This makes it especially useful for morphologically rich languages and noisy social media text.

[↑ Back to Table of Contents](#top_section)


In [96]:
from gensim.models import FastText as GensimFastText

# Train a FastText model on the disaster tweets corpus
# (In production, use a pre-trained model: fasttext.cc/docs/en/crawl-vectors.html)
tokenized_corpus = train_df["lemmatized"].tolist()  # list of token lists

fasttext_model = GensimFastText(
    sentences=tokenized_corpus,
    vector_size=100,   # embedding dimension
    window=5,          # context window size
    min_count=2,       # ignore tokens that appear < 2 times
    workers=4,         # parallel threads
    epochs=10,
    sg=1               # Skip-gram (better for rare/morphological words)
)

print(f"FastText vocabulary : {len(fasttext_model.wv):,} words")
print(f"Vector dimension    : {fasttext_model.wv.vector_size}")
print(f"\nOOV example — 'disastr' (typo): {fasttext_model.wv['disastr'][:6].round(4)}")


FastText vocabulary : 6,497 words
Vector dimension    : 100

OOV example — 'disastr' (typo): [-0.3752  0.3257 -0.184   0.3111  0.119  -0.1248]


In [97]:
# FastText handles OOV words via subword n-grams — no KeyError
print("FastText similarities (trained on disaster tweets):")
pairs = [("fire", "fires"), ("flood", "flooding"), ("help", "helping")]
for w1, w2 in pairs:
    try:
        sim = fasttext_model.wv.similarity(w1, w2)
        print(f"  {w1} vs {w2:10} : {sim:.4f}")
    except KeyError as e:
        print(f"  {e}")

# Demonstrate OOV handling (FastText subword magic)
oov_word = "earthquakke"  # intentional typo
vec = fasttext_model.wv[oov_word]
print(f"\nOOV word '{oov_word}' still gets a vector: {vec[:6].round(4)}")


FastText similarities (trained on disaster tweets):
  fire vs fires      : 0.9650
  flood vs flooding   : 0.8749
  help vs helping    : 0.9135

OOV word 'earthquakke' still gets a vector: [-0.0785  0.1823  0.0118  0.2017  0.0187 -0.0743]


In [98]:
def get_fasttext_vector(tokens, model, dim: int = 100) -> np.ndarray:
    """
    Average FastText word vectors for a document.
    Handles OOV words natively — no zero fallback needed.
    """
    words = tokens if isinstance(tokens, list) else tokens.split()
    if not words:
        return np.zeros(dim)
    vecs = [model.wv[w] for w in words]   # FastText never raises KeyError
    return np.mean(vecs, axis=0)

embeddings_ft = np.vstack([
    get_fasttext_vector(tokens, fasttext_model)
    for tokens in train_df["lemmatized"]
])

print(f"FastText embedding matrix: {embeddings_ft.shape[0]} docs × {embeddings_ft.shape[1]} dims")
print(f"Sample (first 8 values)  : {embeddings_ft[0][:8].round(4)}")

del embeddings_ft  # free memory


FastText embedding matrix: 7613 docs × 100 dims
Sample (first 8 values)  : [-0.192   0.2765 -0.0804  0.2537  0.0357  0.0069 -0.3498  0.297 ]


<a id="Advanced_methods"></a>
<a id="BERT"></a>

## 4.6 BERT — Bidirectional Encoder Representations from Transformers

**What it does:** BERT ([Devlin et al., 2018](https://arxiv.org/abs/1810.04805)) is a deep Transformer model that produces **contextualized word embeddings** — the same word gets a different vector depending on its surrounding context.

**Why it matters:** Word2Vec, GloVe, and FastText produce a single fixed vector per word regardless of context. BERT solves this: `"bank"` in `"river bank"` and `"bank account"` gets two completely different vectors. This is why BERT achieves state-of-the-art results on most NLP benchmarks.

**Architecture overview:**
- **Bidirectional** — reads text left-to-right AND right-to-left simultaneously
- **Pre-trained** on Wikipedia + BooksCorpus using Masked LM + Next Sentence Prediction
- **BERT Base** — 12 layers, 12 attention heads, 110M parameters
- **BERT Large** — 24 layers, 16 attention heads, 340M parameters

**Tokenization:** BERT uses its own **WordPiece** tokenizer — do NOT apply standard NLTK tokenization before feeding text to BERT.

[↑ Back to Table of Contents](#top_section)


In [99]:
!pip install -q transformers tensorflow tensorflow-hub tensorflow-text

import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text  # required by the BERT preprocessor layer

from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

print(f"TensorFlow version : {tf.__version__}")
print(f"Tokenizer vocab sz : {tokenizer.vocab_size:,}")


TensorFlow version : 2.12.0
Tokenizer vocab sz : 30,522


In [100]:
# ── HuggingFace approach — tokenize + encode with bert-base-uncased ────────────
def bert_encode_hf(texts: list, tokenizer, max_len: int = 128):
    """
    Encode a list of texts into BERT-compatible numpy arrays using
    the HuggingFace BertTokenizer.

    Args:
        texts     : List of raw text strings.
        tokenizer : Loaded BertTokenizer instance.
        max_len   : Max sequence length (truncates longer inputs).

    Returns:
        Tuple of numpy arrays (input_ids, attention_masks, token_type_ids),
        each of shape (n_texts, max_len).
    """
    all_ids, all_masks, all_segments = [], [], []

    for text in texts:
        tokens    = tokenizer.tokenize(str(text))
        tokens    = tokens[: max_len - 2]          # leave room for [CLS] + [SEP]
        input_seq = ["[CLS]"] + tokens + ["[SEP]"]
        pad_len   = max_len - len(input_seq)

        ids      = tokenizer.convert_tokens_to_ids(input_seq) + [0] * pad_len
        masks    = [1] * len(input_seq) + [0] * pad_len
        segments = [0] * max_len

        all_ids.append(ids)
        all_masks.append(masks)
        all_segments.append(segments)

    return np.array(all_ids), np.array(all_masks), np.array(all_segments)


# Quick sanity check on 3 samples
sample_texts = train_df["text"].head(3).tolist()
ids, masks, segs = bert_encode_hf(sample_texts, tokenizer, max_len=64)
print(f"input_ids shape      : {ids.shape}")
print(f"attention_mask shape : {masks.shape}")
print(f"token_type_ids shape : {segs.shape}")
print(f"\nSample token IDs (first 15): {ids[0][:15]}")


input_ids shape      : (3, 64)
attention_mask shape : (3, 64)
token_type_ids shape : (3, 64)

Sample token IDs (first 15): [  101  2256 15616  2024  1996  3114  1997  2023  1001  8372  2089 16455
  9641  2149  2035]


In [101]:
# ── TensorFlow Hub approach — full preprocessor + encoder pipeline ────────────
BERT_PREPROCESS_URL = "https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3"
BERT_ENCODER_URL    = "https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4"

bert_preprocessor = hub.KerasLayer(BERT_PREPROCESS_URL, name="bert_preprocessor")
bert_encoder      = hub.KerasLayer(BERT_ENCODER_URL, trainable=False, name="bert_encoder")

print("BERT-Base (12L/768H/12A) loaded successfully.")
print(f"Preprocessor : {BERT_PREPROCESS_URL.split('/')[-2]}")
print(f"Encoder      : {BERT_ENCODER_URL.split('/')[-2]}")


BERT-Base (12L/768H/12A) loaded successfully.
Preprocessor : bert_en_uncased_preprocess
Encoder      : bert_en_uncased_L-12_H-768_A-12


In [102]:
def bert_encode_tfhub(texts, preprocessor, encoder):
    """
    Encode texts using the TF Hub BERT preprocessor + encoder pipeline.

    Args:
        texts       : 1-D string tensor or numpy array of raw text.
        preprocessor: Loaded TF Hub preprocessor KerasLayer.
        encoder     : Loaded TF Hub BERT encoder KerasLayer.

    Returns:
        pooled_output : Tensor of shape (n, 768) — [CLS] sentence embeddings.
        sequence_out  : Tensor of shape (n, seq_len, 768) — per-token embeddings.
    """
    encoder_inputs = preprocessor(texts)
    outputs        = encoder(encoder_inputs)
    return outputs["pooled_output"], outputs["sequence_output"]


# Encode a small sample (full dataset encoding is GPU-recommended)
sample_texts = tf.constant(train_df["text"].head(5).tolist())
pooled, sequence = bert_encode_tfhub(sample_texts, bert_preprocessor, bert_encoder)

print(f"BERT input keys    : {list(bert_preprocessor(sample_texts).keys())}")
print(f"Pooled output shape: {pooled.shape}    ← sentence embedding ([CLS] token)")
print(f"Sequence out shape : {sequence.shape}  ← per-token embeddings")

print(f"\nSample sentence    : {train_df['text'].iloc[0]}")
print(f"CLS embedding (8)  : {pooled[0][:8].numpy().round(4)}")


BERT input keys    : ['input_word_ids', 'input_type_ids', 'input_mask']
Pooled output shape: (5, 768)    ← sentence embedding ([CLS] token)
Sequence out shape : (5, 128, 768)  ← per-token embeddings

Sample sentence    : Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
CLS embedding (8)  : [-0.8219 -0.4706 -0.5762  0.5994 -0.0018 -0.0315  0.6198  0.2176]


<a id="Comparison"></a>
## 4.7 Comparison of Feature Extraction Methods

[↑ Back to Table of Contents](#top_section)


| Method | Type | Captures Syntax | Captures Semantics | Handles OOV | Notes |
|--------|------|:-:|:-:|:-:|-------|
| **BoW / CountVectorizer** | Sparse | ❌ | ❌ | ✅ | Fast, interpretable, ignores order |
| **TF-IDF** | Sparse | ❌ | ❌ | ✅ | Downweights common words automatically |
| **Word2Vec** | Dense | ✅ | ✅ | ❌ | Fixed vectors per word; fast inference |
| **GloVe (pre-trained)** | Dense | ✅ | ✅ | ❌ | Trained on huge corpora; often excellent baseline |
| **FastText** | Dense | ✅ | ✅ | ✅ | Best for morphologically rich / noisy text |
| **BERT** | Contextual | ✅ | ✅✅ | ✅ | Handles polysemy; expensive but most accurate |

### When to use each:

| Task | Recommended Representation |
|------|---------------------------|
| Simple text classification | TF-IDF + logistic regression |
| Similarity search | Word2Vec / GloVe sentence averages |
| Noisy/social media text | FastText |
| State-of-the-art accuracy | Fine-tuned BERT / RoBERTa |
| Low resource / fast iteration | TF-IDF or BoW |


<a id="References"></a>

---
# 5. 📚 References

### Papers
- [Text Classification Algorithms: A Survey](https://arxiv.org/abs/1904.08067) — Kowsari et al., 2019
- [Distributed Representations of Words and Phrases (Word2Vec)](https://papers.nips.cc/paper/5021-distributed-representations-of-words-and-phrases-and-their-compositionality.pdf) — Mikolov et al., 2013
- [GloVe: Global Vectors for Word Representation](https://nlp.stanford.edu/pubs/glove.pdf) — Pennington et al., 2014
- [Enriching Word Vectors with Subword Information (FastText)](https://arxiv.org/abs/1607.04606) — Bojanowski et al., 2017
- [BERT: Pre-training of Deep Bidirectional Transformers](https://arxiv.org/abs/1810.04805) — Devlin et al., 2018

### Books
- [Natural Language Processing in Action](https://www.manning.com/books/natural-language-processing-in-action) — Hobson Lane et al.
- [Speech and Language Processing](https://web.stanford.edu/~jurafsky/slp3/ed3book.pdf) — Jurafsky & Martin

### Kaggle Notebooks & Blogs
- [Getting Started with Text Preprocessing](https://www.kaggle.com/sudalairajkumar/getting-started-with-text-preprocessing)
- [BERT for Humans](https://www.kaggle.com/abhinand05/bert-for-humans-tutorial-baseline-version-2)
- [NLP with Disaster Tweets — EDA, Cleaning & BERT](https://www.kaggle.com/gunesevitan/nlp-with-disaster-tweets-eda-cleaning-and-bert)
- [NLP EDA — BoW, TF-IDF, GloVe, BERT](https://www.kaggle.com/vbmokin/nlp-eda-bag-of-words-tf-idf-glove-bert)
- [In-Depth Guide to Google's BERT](https://www.kaggle.com/ratan123/in-depth-guide-to-google-s-bert)
- [Natural Language Processing Pipeline](https://towardsdatascience.com/natural-language-processing-pipeline-93df02ecd03f)
- [Stemming, Lemmatization: What?](http://hunterheidenreich.com/blog/stemming-lemmatization-what/)

---

<div align="center">
  <h3>⭐ If this notebook was helpful, please upvote! ⭐</h3>
  <p>Contributions, corrections, and suggestions are welcome.</p>
  <a href="#top_section">↑ Back to Top</a>
</div>
